# Auditing the Author Clustering

*Part 2 of 3 in our evaluation of the [What Is Seattle Reading?](https://melaniewalsh.github.io/whats-seattle-reading-viewer/) viewer.*

In Part 1, we evaluated how the viewer groups *titles* into works. This notebook evaluates the other clustering system in the project: how the data pipeline groups *author name spellings* into authors.

Here's the problem it solves. Library catalog data spells the same person's name many different ways. Search the raw data for Octavia Butler and you'll find three of her:

| Raw name in the catalog | Checkouts |
|---|---|
| Octavia E. Butler | 10,630 |
| Butler, Octavia E. | 1,637 |
| Octavia Butler | 303 |

If we don't merge these, every chart about Butler is quietly wrong — her readership gets split into fragments. So the pipeline (`data-prep/cluster_authors.py`) builds a giant mapping from raw spellings to canonical names: about **450,000 mappings**. This kind of task — deciding when two records refer to the same real-world entity — is called **entity resolution**, and it's one of the oldest, hardest problems in information science.

Our question: **how trustworthy is that mapping?**

By the end of this notebook, you will be able to:

- Explain why evaluating entity resolution requires **sampling** (and why the sample must be **stratified**)
- Run a small, honest **annotation task** with a partner and measure your agreement with **Cohen's kappa**
- Estimate **recall** against a curated gold standard
- Report results the way reviewers expect: *"precision X% on a stratified sample (κ = ...); recall Y% against a curated gold set"*

## Two Ways to Be Wrong (and Why We Measure Them Differently)

An author map can fail in two directions:

- A **false merge**: two *different* real people mapped to the same canonical name. This silently corrupts every downstream statistic — reader A's checkouts get credited to reader B's author.
- A **false split** (a *missed* merge): the same person left under two spellings. This is visible in the viewer — you search an author and find duplicates — annoying, but at least discoverable.

**Precision** (are the merges we made correct?) catches false merges. **Recall** (did we find all the merges?) catches false splits. And they demand completely different evaluation strategies:

- For **precision**, we can sample from the 450,000 mappings the pipeline *made* and check them by hand.
- For **recall**, we can't sample what the pipeline *didn't do* — you can't draw a random sample of missing merges! So instead we'll borrow a gold standard: the NAAL dataset from Part 1, where humans exhaustively collected each of 93 authors' name variants. If the humans found a variant and our map missed it, that's a measurable miss.

## Setup

In [ ]:
import json
import urllib.request
from pathlib import Path

import pandas as pd
from sklearn.metrics import cohen_kappa_score

pd.options.display.max_colwidth = 80

# This notebook lives in paper/eval/, so the repo root is two levels up
REPO = Path.cwd().parent.parent

## Loading the Author Map

The map is a big JSON dictionary: raw spelling → canonical name. Let's load it and look around.

In [ ]:
author_map = json.loads((REPO / "data-prep" / "author_map.json").read_text())
print(f"{len(author_map):,} mappings")

# Peek at a few
for raw, canonical in list(author_map.items())[:8]:
    print(f"  {raw!r:50} -> {canonical!r}")

💡 **Tip:** Before any formal sampling, just *scroll*. Look up a few authors you know. Do the mappings look sane? Weird? Building intuition first makes you a much better annotator later — you'll know what "normal" looks like, so "abnormal" will jump out.

In [ ]:
# Your turn: look up an author you know. (Keys are raw spellings, values are canonical names.)
[(raw, canon) for raw, canon in author_map.items() if "morrison" in raw.lower()][:10]

## Why Stratify? (Or: Not All Errors Are Equal)

We can't check 450,000 mappings by hand, so we'll check a **sample of 200**. But a *purely random* sample has a problem: the vast majority of authors in a library catalog are rarely-borrowed, so a random sample would be almost all obscure names. An error on an author with 12 checkouts and an error on an author with 50,000 checkouts would count the same — even though the second one distorts the viewer far more.

The fix is a **stratified sample**: divide the mappings into strata (layers) by how popular the canonical author is, then sample 50 from each layer. That way the popular authors — where errors hurt most — are guaranteed a seat at the table, and we can report precision *per stratum* ("the map is 99% right for popular authors, 92% for the long tail" is far more useful than one blended number).

To stratify, we need each author's total checkouts. The viewer's search index — a public file listing every author with their total — gives us exactly that.

⚠️ **Heads up!** The file is hosted on Cloudflare, which blocks requests whose *user agent* looks like a default Python script (you'd get a mysterious `403: Forbidden`). This is common — servers use the user-agent string to filter automated traffic — and the fix is to send a browser-like user agent with our request. If you've done web scraping before, you've met this move.

In [ ]:
SEARCH_INDEX_URL = "https://pub-7559692ae8974bd6a6e7cc54db47a348.r2.dev/search-index.json"

request = urllib.request.Request(
    SEARCH_INDEX_URL,
    headers={"User-Agent": "Mozilla/5.0 (evaluation notebook; whats-seattle-reading-viewer)"},
)
with urllib.request.urlopen(request) as f:
    search_index = json.load(f)

# Each entry is [slug, display_name, total_checkouts]
totals = {name.lower(): total for slug, name, total in search_index}
print(f"{len(totals):,} authors in the search index")
list(totals.items())[:5]

In [ ]:
# Attach a checkout total to every mapping (via its canonical name)
mappings = pd.DataFrame(
    [(raw, canon) for raw, canon in author_map.items()],
    columns=["raw_name", "canonical_name"],
)
mappings["total_checkouts"] = mappings["canonical_name"].str.lower().map(totals)

print(f"{mappings['total_checkouts'].isna().sum():,} mappings have no total (author fell below "
      f"the index cutoff, or names didn't align) — we'll put them in the lowest stratum")
mappings["total_checkouts"] = mappings["total_checkouts"].fillna(0)
mappings.sample(5, random_state=42)

In [ ]:
# Split into 4 strata by checkout volume, then sample 50 from each.
# RANDOM_SEED is fixed so anyone can re-run this cell and get the SAME sample —
# that's what makes the sample (and therefore our numbers) reproducible.
RANDOM_SEED = 42
PER_STRATUM = 50

mappings["stratum"] = pd.qcut(
    mappings["total_checkouts"].rank(method="first"),  # rank first: many ties at 0
    q=4,
    labels=["tail", "low", "mid", "top"],
)

sample = (
    mappings.groupby("stratum", observed=True)
    .apply(lambda g: g.sample(PER_STRATUM, random_state=RANDOM_SEED), include_groups=False)
    .reset_index()
    .drop(columns="level_1")
)

print(sample["stratum"].value_counts())
sample.head()

## Setting Up the Annotation

Now we turn the sample into an annotation spreadsheet. Two of you will *independently* judge every row — independence is the whole point, so **no discussing rows until both of you have finished**. (If you compare notes as you go, our agreement measure becomes meaningless.)

**The question you're answering for each row:** *do the raw name and the canonical name refer to the same real person?*

**Ground rules:**

- **Judge the merge, not the spelling.** If the canonical form looks ugly (odd casing, stray life dates) but it's clearly the same person — that's `correct`. We're auditing identity decisions, not typography.
- **Research is allowed and encouraged.** Search the [SPL catalog](https://seattle.bibliocommons.com/), Wikipedia, or [VIAF](https://viaf.org/) for the name plus a book title. You're a humanities researcher; act like one.
- **Different real people with similar names merged together = `incorrect`.** This is the error we most need to catch. Watch for parent/child author pairs, common names, and "Jr."s.
- **Truly can't tell after a few minutes of looking = `unsure`.** Don't agonize — `unsure` is an honest answer and we report how many there were.
- Use the `notes` column for anything interesting: suspected different-people merges, gold-standard weirdness, funny catalog typos. These notes feed the paper's error discussion.

In [ ]:
annotation_sheet = sample[["stratum", "raw_name", "canonical_name", "total_checkouts"]].copy()
annotation_sheet["annotator_A"] = ""   # correct / incorrect / unsure
annotation_sheet["annotator_B"] = ""   # correct / incorrect / unsure
annotation_sheet["notes"] = ""

annotation_sheet.to_csv("author_map_annotation_sheet.csv", index=False)
print(f"Wrote author_map_annotation_sheet.csv with {len(annotation_sheet)} rows")
print("Import it into a Google Sheet, annotate independently, then come back.")

⏸️ **Pause here.** Go do the annotation (budget a couple of hours — about a minute per row each). Then come back with the filled-in CSV and run the rest of the notebook.

---

## Scoring Part A: Agreement First, Then Precision

Before we trust the labels, we ask: **did the two annotators agree with each other?** If two thoughtful people can't agree on what counts as a correct merge, our "precision" number is built on sand.

The standard measure is **Cohen's kappa (κ)**, which measures agreement *above what chance would produce*. (Why not just percent agreement? If 95% of merges are correct, two annotators who both mindlessly stamp `correct` on everything would "agree" 95% of the time. Kappa corrects for that.) Rough interpretation: κ above 0.8 is very good; 0.6–0.8 is acceptable; below 0.6 means the task itself was ambiguous and we should tighten the guidelines and re-annotate.

⚠️ **Heads up!** We report the kappa from your *independent* labels — before you discussed anything. Agreement measured after discussion is always inflated, and reviewers know it.

In [ ]:
done = pd.read_csv("author_map_annotation_sheet.csv")

# Kappa on the rows where both annotators gave a real label
both = done[done["annotator_A"].isin(["correct", "incorrect", "unsure"])
            & done["annotator_B"].isin(["correct", "incorrect", "unsure"])]
kappa = cohen_kappa_score(both["annotator_A"], both["annotator_B"])
print(f"Cohen's kappa = {kappa:.3f}  (on {len(both)} double-annotated rows)")

disagreements = both[both["annotator_A"] != both["annotator_B"]]
print(f"{len(disagreements)} disagreements to adjudicate:")
disagreements[["stratum", "raw_name", "canonical_name", "annotator_A", "annotator_B"]]

**Now** you can talk to each other. Go through each disagreement together, decide on a final label, and record it in a new column called `final` (copy the agreed label for the rows where you already matched). Then precision is simple arithmetic — with `unsure` rows excluded from the denominator but honestly reported:

In [ ]:
done["final"] = done["final"] if "final" in done else done["annotator_A"]  # replace with adjudicated labels

decided = done[done["final"].isin(["correct", "incorrect"])]
n_unsure = (done["final"] == "unsure").sum()

overall_precision = (decided["final"] == "correct").mean()
print(f"Overall precision: {overall_precision:.3f}  ({len(decided)} decided, {n_unsure} unsure)\n")

print("Precision by stratum:")
decided.groupby("stratum", observed=True)["final"].apply(lambda s: (s == "correct").mean())

## Scoring Part B: Recall Against the NAAL Gold Standard

Precision told us the merges we made are (hopefully!) trustworthy. But how many merges did the pipeline *miss*? For the 93 NAAL authors, the research team hand-collected every raw `Creator` spelling and unified them in `Normalized.Creator`. So for each gold author, we ask: **does our map send all of their known variants to a single canonical name?**

In [ ]:
GOLD_URL = (
    "https://seattle-library-checkout-data.s3.us-west-2.amazonaws.com/"
    "norton-anthology_spl-checkouts_2005-2025.csv"
)
gold = pd.read_csv(GOLD_URL, usecols=["Creator", "Normalized.Creator"], low_memory=False).drop_duplicates()

variants_per_author = gold.groupby("Normalized.Creator")["Creator"].apply(set)
print(f"{len(variants_per_author)} gold authors, "
      f"{variants_per_author.str.len().sum():,} raw variants collected by hand")
variants_per_author.head()

In [ ]:
def resolve(raw_name):
    """Where does our map send this raw spelling? (Names not in the map stay as-is.)"""
    return author_map.get(raw_name, raw_name)

results = []
for gold_author, variants in variants_per_author.items():
    resolved = {resolve(v) for v in variants}
    results.append({
        "gold_author": gold_author,
        "n_variants": len(variants),
        "n_after_resolution": len(resolved),
        "fully_unified": len(resolved) == 1,
    })

recall_df = pd.DataFrame(results)
print(f"Authors fully unified by our map: "
      f"{recall_df['fully_unified'].sum()} / {len(recall_df)} "
      f"({recall_df['fully_unified'].mean():.1%})")

# Variant-level recall: of all hand-collected variants, what share resolved
# to their author's most common canonical name?
recall_df.sort_values("n_after_resolution", ascending=False).head(10)

⚠️ **Heads up!** Before you trust these numbers, check for a boring failure mode: the raw `Creator` strings in the gold CSV must actually match the keys in `author_map.json` — same trailing punctuation, same handling of life dates. If lots of gold variants simply *aren't in the map at all* (as opposed to being mapped somewhere wrong), that's a formatting mismatch between the two datasets, not a pipeline miss — and we'd be under-reporting recall. Spot-check a few misses by hand before writing anything down. If you find a mismatch, flag it — don't quietly patch around it.

## Your Turn: Read the Misses

For the authors that did *not* fully unify, look at what happened. Are the leftover variants misspellings the pipeline was (rightly) too cautious to merge? Life-date formatting issues? Genuinely hard cases like initials-only names? Sort them into piles, count the piles, and pick one or two vivid examples — same drill as Part 1.

## What to Upload

- [ ] This notebook, run top to bottom
- [ ] `author_map_annotation_sheet.csv` with both annotators' labels **and** the adjudicated `final` column
- [ ] The headline sentence, filled in: *"Precision \_\_\_% on a stratified sample of 200 mappings (κ = \_\_\_); \_\_\_% of NAAL gold authors fully unified."*
- [ ] Precision per stratum (the table above)
- [ ] A short written summary of the misses in Part B, plus anything interesting from your `notes` column